## Structured Output

--> Models can be requested to provide their response in a formate matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain support multiple schema types and methods for enforcing structured output 

## Pydantic

--> pydantic models provide the richest feature set with field validation, descriptions, and nested structures. 

In [2]:
import os 
from langchain.chat_models import init_chat_model
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY') #type:ignore
model = init_chat_model("groq:openai/gpt-oss-120b")

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title : str =Field(description="The title of the movie")
    year : int =Field(description="This year the movie was released")
    director : str =Field(description="The director of the movie")
    rating : float =Field(description="The movies rating out of 10")

In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020763589BE0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002076358A660>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The t

In [5]:
model_with_structure.invoke("Provide details about the movie thor ragnarok")

Movie(title='Thor: Ragnarok', year=2017, director='Taika Waititi', rating=7.9)

In [6]:
response = model_with_structure.invoke("Provide details about the movie thor ragnarok")
response

Movie(title='Thor: Ragnarok', year=2017, director='Taika Waititi', rating=7.9)

## Message output alongside parsed structure



In [7]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    """ A movie with details. """
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")
    
model_with_structure = model.with_structured_output(Movie,include_raw=True)
response = model_with_structure.invoke("Provide details about the movie Inception")
response
    

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "User wants details about the movie Inception. We can call function Movie with appropriate parameters. Need director: Christopher Nolan, rating (maybe 8.8), title: Inception, year: 2010. Provide details. We'll call function.", 'tool_calls': [{'id': 'fc_bf4eadb1-09e4-44a2-9636-149caabbc54b', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 165, 'total_tokens': 268, 'completion_time': 0.218166785, 'completion_tokens_details': {'reasoning_tokens': 51}, 'prompt_time': 0.038424801, 'prompt_tokens_details': None, 'queue_time': 0.360146427, 'total_time': 0.256591586}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_803c0ba83d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='

## Nested Structure

In [8]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str
    
class Movie_Details(BaseModel):
    title:str
    year:int
    title:str
    cast : list[Actor]
    genres:list[str]
    budget : float | None = Field(None,description="budget in million USD")

model_with_structure = model.with_structured_output(Movie_Details)

response = model_with_structure.invoke("Provide details about the movie thor")
response

Movie_Details(title='Thor', year=2011, cast=[Actor(name='Chris Hemsworth', role='Thor'), Actor(name='Natalie Portman', role='Jane Foster'), Actor(name='Tom Hiddleston', role='Loki'), Actor(name='Anthony Hopkins', role='Odin'), Actor(name='Stellan Skarsgård', role='Erik Selvig'), Actor(name='Ray Stevenson', role='Volstagg'), Actor(name='Tadanobu Asano', role='Hogun'), Actor(name='Jaimie Alexander', role='Sif'), Actor(name='Clark Gregg', role='Phil Coulson'), Actor(name='Kat Dennings', role='Darcy Lewis')], genres=['Action', 'Adventure', 'Fantasy'], budget=150000000.0)

## TypedDict

--> TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation

In [10]:
from typing_extensions import TypedDict, Annotated
class Movie_Dict(TypedDict):
    """ A movie with details."""
    title : Annotated[str, ..., "The title of the movie"]
    year : Annotated[int, ..., "The year movie was released"]
    director : Annotated[str, ..., "The director of the movie"]
    rating : Annotated[float, ..., "the Movie's rating out of 10"]
    
model_with_typedict = model.with_structured_output(Movie_Dict)
response = model_with_typedict.invoke("Please provide the details of the movie avengers")
response


{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [11]:
from pydantic import BaseModel, Field

class Actor(TypedDict):
    name:str
    role:str
    
class Movie_Details(TypedDict):
    title:str
    year:int
    title:str
    cast : list[Actor]
    genres:list[str]
    budget : float | None = Field(None,description="budget in million USD")

model_with_structure = model.with_structured_output(Movie_Details)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'},
  {'name': 'Clark Gregg', 'role': 'Phil Coulson'},
  {'name': 'Cobie Smulders', 'role': 'Maria Hill'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'The Avengers',
 'year': 2012}

In [13]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

## Data Classes

--> A data is a typically containing data. although there aren't really any restrictions. you create it using the @dataclass decorator

In [27]:
import os 
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY") #type:ignore



In [29]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

class ContactInfo(BaseModel):
    """ Contact information for a person """
    
    name : str = Field(description="The name of the person")
    email : str = Field(description="The email of the person")
    phone : int = Field(description="The phone number of the person")
    
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    
)

agent = create_agent(
    model=llm,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

print(result)

{'messages': [HumanMessage(content='John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='95bfefbf-88e8-4868-9230-c24e83bf62a9'), AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":5551234567}', additional_kwargs={'reasoning_content': 'We need to produce JSON with fields name, email, phone. Phone is integer type. But phone number includes parentheses and hyphens. The schema says type integer. We need to provide an integer. Perhaps remove formatting: 5551234567. So phone: 5551234567. Ensure JSON is compact. Output only JSON.'}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 232, 'total_tokens': 334, 'completion_time': 0.105174203, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.011405772, 'prompt_tokens_details': None, 'queue_time': 0.042475202, 'total_time': 0.116579975}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand

In [30]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone=5551234567)

In [31]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

class ContactInfo(BaseModel):
    """ Contact information for a person """
    
    name : str = Field(description="The name of the person")
    email : str = Field(description="The email of the person")
    phone : int = Field(description="The phone number of the person")
    
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    
)

agent = create_agent(
    model=llm,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone=5551234567)

In [33]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

@dataclass
class ContactInfo:
    """ Contact information for a person """
    
    name : str = Field(description="The name of the person")
    email : str = Field(description="The email of the person")
    phone : int = Field(description="The phone number of the person")
    
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    
)

agent = create_agent(
    model=llm,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

result['structured_response']


ContactInfo(name='John Doe', email='john@example.com', phone=5551234567)